# Experiment 2 (Part 2 of 2): LoRA Fine-Tuning of Falcon-7B / GPT-2 for Text-to-SQL

**Course:** HAAI302COM - Artificial Intelligence Laboratory
**Experiment 2:** Fine-tuning a Pre-trained LLM using LoRA (Parameter-Efficient Fine-Tuning)

## Aim
To fine-tune a pre-trained causal language model (Falcon-7B on a GPU, GPT-2 on a CPU) on a domain-specific dataset using Low-Rank Adaptation via the PEFT library, and to evaluate its performance before and after fine-tuning.

## The domain task: text-to-SQL
We teach the model to write SQL. Each example has three parts:
- a **schema** (`CREATE TABLE ...`),
- a **question** in English,
- the correct **SQL query**.

This is a good fine-tuning task because (a) a general base model is poor at it, (b) the output has a strict format the model must learn, and (c) we can grade answers objectively by comparing the generated SQL with the correct SQL.

Dataset: [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context), about 78,000 examples built from WikiSQL and Spider.

## How to run this notebook
- **Colab GPU (recommended, intended setup):** *Runtime > Change runtime type > T4 GPU*. The notebook detects the GPU and fine-tunes **Falcon-7B** in 4-bit (QLoRA). Expect about 20 to 30 minutes of training.
- **CPU only:** the same code automatically falls back to **GPT-2 small** with a smaller training set. Expect 10 to 20 minutes on a modern laptop.

Every cell works in both modes; only the settings in Section 0 change. Part 1 (`01_LoRA_Anatomy_with_PEFT.ipynb`) explains what LoRA adds to the model; read it first.

## Workflow
`load data -> format prompts -> tokenize with label masking -> load base model -> evaluate BEFORE -> attach LoRA -> train -> evaluate AFTER -> save / reload / merge adapter`

## 0. Setup and the one switch that matters

`USE_GPU` decides everything else:

| Setting | GPU (Colab T4) | CPU fallback |
|---|---|---|
| `BASE_MODEL` | `tiiuae/falcon-7b` | `gpt2` |
| loading | 4-bit NF4 quantization (QLoRA) | full 32-bit |
| `TARGET_MODULES` | `query_key_value` (Falcon's fused attention projection) | `c_attn`, `c_proj` |
| training examples | 3,000 | 1,500 |
| `LEARNING_RATE` | 2e-4 | 1e-3 |

LoRA uses a **higher learning rate** than full fine-tuning (which typically uses 1e-5 to 5e-5), because only the tiny A and B matrices move and B starts at zero. Small models tolerate even higher rates than 7B models.

In [ ]:
# !pip install -q transformers peft datasets accelerate bitsandbytes pandas matplotlib
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import transformers, huggingface_hub
transformers.logging.set_verbosity_error()
huggingface_hub.utils.logging.set_verbosity_error()

import math, re, time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

pd.set_option("display.max_colwidth", 100)
USE_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if USE_GPU else "none, using the CPU fallback")

In [ ]:
if USE_GPU:
    BASE_MODEL, TARGET_MODULES = "tiiuae/falcon-7b", ["query_key_value"]
    N_TRAIN, LEARNING_RATE, BATCH, GRAD_ACCUM = 3000, 2e-4, 4, 4
else:
    BASE_MODEL, TARGET_MODULES = "gpt2", ["c_attn", "c_proj"]
    N_TRAIN, LEARNING_RATE, BATCH, GRAD_ACCUM = 1500, 1e-3, 8, 2
N_VAL, N_TEST, MAX_LEN = 200, 50, 256        # validation for loss, test for generated-SQL accuracy
ADAPTER_DIR = f"lora-sql-{BASE_MODEL.split('/')[-1]}"
pd.Series({"BASE_MODEL": BASE_MODEL, "TARGET_MODULES": TARGET_MODULES, "N_TRAIN": N_TRAIN,
           "LEARNING_RATE": LEARNING_RATE, "effective batch": BATCH * GRAD_ACCUM}).to_frame("value")

**What to notice.** The *effective batch size* is `BATCH x GRAD_ACCUM`. Gradient accumulation runs several small batches and adds up their gradients before one optimizer step, so we get the stability of a batch of 16 without holding 16 examples in GPU memory at once.

## 1. Load and split the dataset

`load_dataset` downloads the data as a `Dataset` object (a table backed by Apache Arrow). We shuffle once with a fixed seed and cut three non-overlapping slices:
- **train** - the examples the model learns from,
- **validation** - used for the loss curve during training,
- **test** - never seen in training; used for the before/after accuracy.

In [ ]:
raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
train_raw = raw.select(range(N_TRAIN))
val_raw   = raw.select(range(N_TRAIN, N_TRAIN + N_VAL))
test_raw  = raw.select(range(N_TRAIN + N_VAL, N_TRAIN + N_VAL + N_TEST))
print(raw)
pd.DataFrame(train_raw[:3])[["context", "question", "answer"]]

**What to notice.** `context` holds only the `CREATE TABLE` line (column names and types), not real rows. The model must map English words like "older than 56" to SQL like `WHERE age > 56`, using the column names it is given.

## 2. A fixed prompt template

The model needs one consistent format so it learns *where* the SQL starts and stops. We use `###` headers; the SQL comes after `### SQL:` and ends with the model's end-of-sequence token. The **same template** is used for training, for evaluation before fine-tuning and for evaluation after it, so the comparison is fair.

In [ ]:
def build_prompt(example):
    return f"### Schema:\n{example['context']}\n### Question:\n{example['question']}\n### SQL:\n"

print(build_prompt(train_raw[0]) + train_raw[0]["answer"])

## 3. Tokenization with label masking

For causal LM training, the `labels` are the same token IDs as the input; the model learns to predict each next token. But we do **not** want to train the model to predict the schema and question (those are given at test time). We only want it to learn the SQL.

So for every example:
1. Tokenize the prompt and the answer separately.
2. `input_ids` = prompt tokens + answer tokens + end-of-sequence.
3. `labels` = `-100` for every prompt position, then the answer tokens. PyTorch's loss function **ignores** positions labelled `-100`.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token          # neither GPT-2 nor Falcon ships a pad token

def tokenize(example):
    prompt_ids = tok(build_prompt(example))["input_ids"]
    answer_ids = tok(example["answer"])["input_ids"] + [tok.eos_token_id]   # teach it to stop
    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]              # mask the prompt
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

In [ ]:
train_ds = train_raw.map(tokenize, remove_columns=train_raw.column_names)
val_ds = val_raw.map(tokenize, remove_columns=val_raw.column_names)
lengths = pd.Series([len(x) for x in train_ds["input_ids"]])
lengths.describe()[["mean", "50%", "max"]].round(1).to_frame("tokens per example")

**What to notice.** Examples are short (mostly under 100 tokens), so `MAX_LEN=256` truncates almost nothing. Short sequences are one reason this experiment is cheap.

### 3.1 Look at one tokenized example, token by token

This table is the most important check in the notebook. If the masking is wrong, the model learns the wrong thing and nothing later will tell you why.

In [ ]:
ex = train_ds[0]
view = pd.DataFrame({"token": tok.convert_ids_to_tokens(ex["input_ids"]), "input_id": ex["input_ids"], "label": ex["labels"]})
view.tail(15)

**What to notice.** Rows from the prompt (up to and including the `SQL:` header and its newline) have `label = -100`. From the first SQL token onwards the label equals the input ID, and the very last row is the end-of-sequence token with a real label: the model is trained to *stop* after the query.

### 3.2 The data collator pads each batch

Examples have different lengths. `DataCollatorForSeq2Seq` pads `input_ids` with the pad token, pads `attention_mask` with 0, and pads `labels` with `-100`, so padding never contributes to the loss. (Despite its name, it works for causal models too; the plain language-modelling collator would overwrite our carefully masked labels.)

In [ ]:
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
demo = collator([train_ds[0], train_ds[1]])
pd.DataFrame({k: [tuple(v.shape)] for k, v in demo.items()}, index=["batch shape"])

## 4. Load the base model

On the GPU we load Falcon-7B in **4-bit** with `BitsAndBytesConfig`:
- `load_in_4bit=True` - store frozen weights in 4 bits instead of 16, about 3.5 GB instead of 14 GB.
- `bnb_4bit_quant_type="nf4"` - "NormalFloat 4", a 4-bit format designed for normally distributed weights.
- `bnb_4bit_compute_dtype=torch.float16` - weights are expanded to 16-bit on the fly for each matrix multiply. (The T4 does not support bfloat16.)
- `bnb_4bit_use_double_quant=True` - also compresses the quantization constants, saving another few hundred MB.

4-bit frozen base + LoRA adapters trained in 16-bit is exactly the **QLoRA** recipe. `prepare_model_for_kbit_training` casts layer norms to 32-bit for stability and turns on gradient checkpointing (recompute activations during the backward pass instead of storing them, trading time for memory).

In [ ]:
if USE_GPU:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
print(f"parameters: {model.num_parameters() / 1e6:,.0f} M")
print(f"memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

**What to notice.** On Colab, Falcon-7B reports about 7,000 M parameters but a memory footprint of only about 4 GB, thanks to 4-bit storage. GPT-2 on CPU uses about 0.5 GB.

## 5. Evaluation tools (used before AND after training)

We measure two things:

1. **Validation loss / perplexity** on the SQL tokens only (thanks to label masking). Perplexity = e raised to the loss. It measures how surprised the model is by the correct SQL. Lower is better.
2. **Generated-SQL accuracy** on the test set. The model writes a query with greedy decoding, we cut it at the first newline (every gold answer is one line), and compare it with the correct SQL:
   - **exact match** - identical after normalising spaces, case and a trailing semicolon,
   - **token F1** - partial credit: the overlap of SQL tokens between prediction and answer (like comparing two bags of words).

In [ ]:
def normalize_sql(s):
    return " ".join(s.strip().rstrip(";").lower().split())

def token_f1(pred, gold):
    p, g = normalize_sql(pred).split(), normalize_sql(gold).split()
    common = sum(min(p.count(t), g.count(t)) for t in set(p))
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

In [ ]:
@torch.no_grad()
def generate_sql(m, examples, batch_size=8):
    m.eval()
    tok.padding_side = "left"                               # left padding for generation
    preds = []
    for i in range(0, len(examples), batch_size):
        batch = examples.select(range(i, min(i + batch_size, len(examples))))
        enc = tok([build_prompt(e) for e in batch], return_tensors="pt", padding=True).to(m.device)
        out = m.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        preds += [t.strip().split("\n")[0] for t in texts]  # keep only the first line
    tok.padding_side = "right"                              # back to right padding for training
    return preds

In [ ]:
def score(preds, examples):
    golds = examples["answer"]
    return {"exact match %": round(100 * sum(normalize_sql(p) == normalize_sql(g) for p, g in zip(preds, golds)) / len(golds), 1),
            "token F1 %": round(100 * sum(token_f1(p, g) for p, g in zip(preds, golds)) / len(golds), 1)}

## 6. Evaluate the base model BEFORE fine-tuning

We give the untouched base model the same prompts. A base model has seen some SQL on the internet, but it was never taught this format.

In [ ]:
t0 = time.time()
preds_before = generate_sql(model, test_raw)
scores_before = score(preds_before, test_raw)
print(scores_before, f"| {time.time() - t0:.0f}s")

In [ ]:
pd.DataFrame({"question": test_raw["question"][:5], "gold SQL": test_raw["answer"][:5], "base model output": preds_before[:5]})

**What to notice.** Exact match is near 0%. The base model typically repeats the question, continues with another `###` header, writes prose, or produces SQL-like text with invented column names. Token F1 is above zero only because words like `SELECT` or a column name appear by chance.

## 7. Attach LoRA adapters

The same `LoraConfig` you studied in Part 1:
- `r=16`, `lora_alpha=32` - rank 16, scaling 2.
- `target_modules` - from Section 0.
- `lora_dropout=0.05` - light regularisation on the adapter input.
- `task_type="CAUSAL_LM"`.

After wrapping, only the adapter parameters require gradients.

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=TARGET_MODULES, lora_dropout=0.05,
                         bias="none", task_type="CAUSAL_LM",
                         fan_in_fan_out=(BASE_MODEL == "gpt2"))   # GPT-2 stores Conv1D weights transposed
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

**What to notice.** For Falcon-7B with `query_key_value` and r=16, about 4.7 M trainable parameters out of about 7 billion: under 0.1%. For GPT-2 with `c_attn` and `c_proj`, about 1.6 M out of 126 M.

## 8. Training arguments

`TrainingArguments` holds every knob of the training loop. The ones we set:

| Argument | Value | Why |
|---|---|---|
| `num_train_epochs` | 1 | one pass over 1,500 to 3,000 examples is enough for a format-learning task |
| `per_device_train_batch_size` | `BATCH` | examples per forward pass |
| `gradient_accumulation_steps` | `GRAD_ACCUM` | effective batch = `BATCH x GRAD_ACCUM` |
| `learning_rate` | `LEARNING_RATE` | LoRA tolerates high rates |
| `lr_scheduler_type` | `"cosine"` | start high, decay smoothly towards zero |
| `warmup_steps` | 10 | ramp up the rate at first so early noisy gradients do not wreck the adapter |
| `eval_strategy`, `eval_steps` | `"steps"`, 25 | compute validation loss every 25 optimizer steps |
| `logging_steps` | 10 | record training loss every 10 steps |
| `fp16` | `True` on GPU | 16-bit mixed precision: faster, half the activation memory |
| `save_strategy` | `"no"` | we save the adapter ourselves at the end |
| `report_to` | `"none"` | no external experiment tracker |

In [ ]:
args = TrainingArguments(
    output_dir="lora-sql-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    eval_strategy="steps", eval_steps=25,
    logging_steps=10,
    fp16=USE_GPU,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)
print("optimizer steps:", math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM)))

### 8.1 Validation loss at step 0

Before training, the LoRA model is identical to the base model (B is zero, see Part 1, Section 4.2). So `trainer.evaluate()` now gives the **base model's** validation loss on the SQL tokens.

In [ ]:
eval_before = trainer.evaluate()
print(f"validation loss before: {eval_before['eval_loss']:.3f} | perplexity: {math.exp(eval_before['eval_loss']):.1f}")

## 9. Train

In [ ]:
t0 = time.time()
train_result = trainer.train()
print(f"training time: {(time.time() - t0) / 60:.1f} min | final training loss: {train_result.training_loss:.3f}")

### 9.1 Read the loss curve from the trainer's log

`trainer.state.log_history` is a list of dictionaries: one per logging event (with `loss`) and one per evaluation (with `eval_loss`). Turning it into a DataFrame lets us plot both curves.

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
train_curve = log.dropna(subset=["loss"])[["step", "loss", "learning_rate"]]
val_curve = log.dropna(subset=["eval_loss"])[["step", "eval_loss"]]
val_curve.assign(perplexity=val_curve["eval_loss"].apply(math.exp).round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_curve["step"], train_curve["loss"], label="training loss")
ax[0].plot(val_curve["step"], val_curve["eval_loss"], "o-", label="validation loss")
ax[0].set_xlabel("optimizer step"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].plot(train_curve["step"], train_curve["learning_rate"])
ax[1].set_xlabel("optimizer step"); ax[1].set_ylabel("learning rate"); ax[1].set_title("warmup + cosine decay")
plt.tight_layout(); plt.show()

**What to notice.**
- The loss falls steeply in the first few dozen steps: that is the model learning the **format** (where SQL starts, `SELECT ... FROM ... WHERE`, and when to stop). The slower decline afterwards is the model learning the **mapping** from English phrases to conditions.
- Training and validation loss stay close together, so the adapter is not memorising the training set. If validation loss started rising while training loss kept falling, that would be overfitting: reduce epochs, `r` or the learning rate.
- The right plot shows the learning rate ramping up during warmup and then following a cosine curve towards zero.

## 10. Evaluate AFTER fine-tuning

Same test examples, same prompts, same greedy decoding. Only the adapter is new.

In [ ]:
preds_after = generate_sql(model, test_raw)
scores_after = score(preds_after, test_raw)
eval_after = trainer.evaluate()
comparison = pd.DataFrame({
    "before LoRA": {**scores_before, "validation perplexity": round(math.exp(eval_before["eval_loss"]), 2)},
    "after LoRA": {**scores_after, "validation perplexity": round(math.exp(eval_after["eval_loss"]), 2)},
})
comparison

In [ ]:
pd.DataFrame({"question": test_raw["question"][:8], "gold SQL": test_raw["answer"][:8],
              "before": preds_before[:8], "after": preds_after[:8]})

**What to notice.**
- Exact match jumps from near 0% to a substantial share of the test set. Falcon-7B reaches much higher accuracy than GPT-2, because the bigger model already understands English questions well; LoRA only teaches it the format and the mapping. Typical results are roughly 60 to 80% exact match for Falcon-7B and roughly 20 to 40% for GPT-2 after one epoch, but your numbers will vary.
- Validation perplexity on the SQL tokens drops sharply.
- We changed well under 1% of the parameters, so the improvement comes from a few MB of adapter weights.

### 10.1 Error analysis: what does the model still get wrong?

Looking at the failures tells us more than the score. Common error types in text-to-SQL:
- wrong **column** chosen (similar names),
- wrong **literal** (the value is copied with a different case or spelling),
- wrong **aggregation** (`COUNT` vs `SUM`) or a missing condition.

In [ ]:
errors = pd.DataFrame({"question": test_raw["question"], "gold": test_raw["answer"], "pred": preds_after})
errors["F1"] = [round(token_f1(p, g), 2) for p, g in zip(errors["pred"], errors["gold"])]
errors = errors[[normalize_sql(p) != normalize_sql(g) for p, g in zip(errors["pred"], errors["gold"])]]
errors.sort_values("F1", ascending=False).head(8)

**What to notice.** Many "wrong" answers have high token F1: they differ only in a literal value (`'California'` vs `'california'`) or in quoting, and would still return the correct rows. Exact match is a strict metric. A stronger evaluation would execute both queries on a real database and compare the results (*execution accuracy*).

## 11. Save the adapter, then reload it on a fresh base model

`model.save_pretrained` writes only the adapter. To use it later, load the base model again and attach the adapter with `PeftModel.from_pretrained`.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
sizes = {f: round(os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6, 2) for f in os.listdir(ADAPTER_DIR)}
pd.Series(sizes).to_frame("size (MB)")

**What to notice.** `adapter_model.safetensors` is tens of MB for Falcon-7B (and a few MB for GPT-2), compared with about 14 GB for the full Falcon-7B checkpoint in 16-bit.

In [ ]:
del trainer, model                       # free memory before loading a second copy
if USE_GPU:
    torch.cuda.empty_cache()
    fresh_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
else:
    fresh_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
reloaded = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)
print("reloaded scores:", score(generate_sql(reloaded, test_raw), test_raw))

**What to notice.** The reloaded model gives the same scores as Section 10. The adapter file alone fully captures what training learned.

### 11.1 Try your own question

In [ ]:
my_example = {"context": "CREATE TABLE students (name VARCHAR, branch VARCHAR, cgpa REAL)",
              "question": "List the names of Computer branch students with CGPA above 8.5"}
from datasets import Dataset
print(generate_sql(reloaded, Dataset.from_list([my_example]))[0])

## 12. Merge for deployment (GPT-2 path)

`merge_and_unload()` folds the adapter into the base weights and returns a plain model with no PEFT dependency at inference time.

For Falcon-7B, do **not** merge into the 4-bit model: rounding the merged weights back to 4 bits loses accuracy. The standard practice is to reload the base in 16-bit on a machine with enough memory (about 15 GB), attach the adapter, then merge. That does not fit comfortably on a free T4, so this cell only merges on the GPT-2 path.

In [ ]:
if not USE_GPU:
    merged = reloaded.merge_and_unload()
    merged.save_pretrained("gpt2-sql-merged")
    print(type(merged).__name__, "| merged scores:", score(generate_sql(merged, test_raw), test_raw))
else:
    print("Skipped on the 4-bit Falcon path: merge on a machine that can load the base model in float16.")

## 13. Optional parameter experiment: does rank matter?

Set `RUN_RANK_EXPERIMENT = True` to train three short runs that differ only in `r` (keeping `lora_alpha = 2r`), on 500 examples each. This is slow (three trainings), so it is off by default. Record the results in the summary table.

In [ ]:
RUN_RANK_EXPERIMENT = False

def quick_run(r, n_train=500):
    if USE_GPU:
        base = prepare_model_for_kbit_training(
            AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto"))
    else:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    m = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, target_modules=TARGET_MODULES,
                                        lora_dropout=0.05, task_type="CAUSAL_LM",
                                        fan_in_fan_out=(BASE_MODEL == "gpt2")))
    t = Trainer(model=m, args=args, train_dataset=train_ds.select(range(n_train)), eval_dataset=val_ds, data_collator=collator)
    t.train()
    return {"r": r, "trainable": m.get_nb_trainable_parameters()[0], "val loss": round(t.evaluate()["eval_loss"], 3),
            **score(generate_sql(m, test_raw), test_raw)}

In [ ]:
if RUN_RANK_EXPERIMENT:
    display(pd.DataFrame([quick_run(r) for r in [2, 8, 32]]))
else:
    print("Rank experiment skipped. Set RUN_RANK_EXPERIMENT = True to run it.")

**What to expect.** Going from r=2 to r=8 usually helps a little; going from 8 to 32 usually changes very little for a narrow task like this, while doubling the adapter size. The task needs only a "small" change to the model, which is exactly the assumption behind LoRA.

## 14. Results summary

| Metric | Before LoRA | After LoRA |
|---|---|---|
| Exact match % | | |
| Token F1 % | | |
| Validation perplexity | | |
| Trainable parameters | (none) | |
| Adapter size (MB) | | |
| Training time (min) | | |

## Conclusion
LoRA adapted a general pre-trained model to the text-to-SQL domain by training well under 1% of its parameters. The base model could not follow the format; after one epoch the fine-tuned model produces valid, mostly correct SQL, with a large drop in validation perplexity. With 4-bit quantization (QLoRA), a 7-billion-parameter model can be fine-tuned on a single free GPU, and the result is a small adapter file that can be switched on, off, or merged into the base model.

## Viva questions
1. Why are the prompt tokens labelled `-100`? What would the model learn if they were not?
2. Why does LoRA use a higher learning rate than full fine-tuning?
3. What do QLoRA's two halves ("Q" and "LoRA") each contribute?
4. Why is the validation loss at step 0 equal to the base model's loss?
5. Why is exact match a pessimistic metric for SQL, and what would be better?
6. Why should you not merge an adapter into a 4-bit quantized model?

## Exercise
Fine-tune a second adapter on a different domain (for example, `medalpaca/medical_meadow_medical_flashcards`) on the same base model. Load both adapters with `model.load_adapter(path, adapter_name=...)` and switch between them with `model.set_adapter(...)`.